In [2]:
import numpy as np
import pandas as pd
import random, os
import matplotlib.pyplot as plt
from collections import defaultdict
from datetime import datetime

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, BaggingClassifier, VotingClassifier, StackingClassifier
from sklearn.svm import SVC
from sklearn.cluster import KMeans

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

import torch
import torch.nn as nn
import torch.optim as optim

from matplotlib.backends.backend_pdf import PdfPages


class Config:
    DATA_DIR="Data"
    OUTPUT_DIR="results"
    RANDOM_STATE=475
    N_SPLITS=5
    CLUSTER_RANGE=range(5,26)


cfg=Config()
os.makedirs(cfg.OUTPUT_DIR,exist_ok=True)


class DataLoader:
    def load(self):
        self.D=pd.read_csv(f"{cfg.DATA_DIR}/disease_similarity.csv",header=None).values
        self.S=pd.read_csv(f"{cfg.DATA_DIR}/snoRNA_similarity.csv",header=None).values
        self.A=pd.read_csv(f"{cfg.DATA_DIR}/known_snoRNA_disease.csv",header=None).values
        return self


class PairBuilder:
    def __init__(self,D,S,A):
        self.D=D; self.S=S; self.A=A
    def split(self):
        pos=[]; neg=[]
        for r in range(self.A.shape[0]):
            for d in range(self.A.shape[1]):
                (pos if self.A[r,d]==1 else neg).append((r,d))
        return pos,neg
    def feat(self,r,d):
        return np.concatenate([self.D[d],self.S[r]])


class ClusterSampler:
    def __init__(self,k):
        self.k=k
    def sample(self,neg,feats,n_pos):
        labels=KMeans(self.k,random_state=cfg.RANDOM_STATE).fit_predict(feats)
        buckets=defaultdict(list)
        for p,l in zip(neg,labels): buckets[l].append(p)
        out=[]
        for b in buckets:
            k=int(len(buckets[b])/len(feats)*n_pos)
            out+=random.sample(buckets[b],max(1,min(k,len(buckets[b]))))
        return out


class DatasetBuilder:
    def __init__(self,pb): self.pb=pb
    def build(self,neg,pos):
        X=[]; y=[]
        for r,d in neg: X.append(self.pb.feat(r,d)); y.append(0)
        for r,d in pos: X.append(self.pb.feat(r,d)); y.append(1)
        return np.array(X),np.array(y)


class DNN(nn.Module):
    def __init__(self,dim):
        super().__init__()
        self.net=nn.Sequential(
            nn.Linear(dim,512),nn.ReLU(),nn.Dropout(0.3),
            nn.Linear(512,256),nn.ReLU(),nn.Dropout(0.3),
            nn.Linear(256,1),nn.Sigmoid()
        )
    def forward(self,x): return self.net(x)


class DNNClassifier:
    def __init__(self,dim):
        self.scaler=StandardScaler()
        self.model=DNN(dim)
        self.opt=optim.Adam(self.model.parameters(),lr=1e-3)
        self.loss=nn.BCELoss()
    def fit(self,X,y,epochs=25):
        X=self.scaler.fit_transform(X)
        X=torch.tensor(X,dtype=torch.float32)
        y=torch.tensor(y,dtype=torch.float32).view(-1,1)
        for _ in range(epochs):
            p=self.model(X)
            l=self.loss(p,y)
            self.opt.zero_grad()
            l.backward()
            self.opt.step()
    def predict_proba(self,X):
        X=self.scaler.transform(X)
        X=torch.tensor(X,dtype=torch.float32)
        return self.model(X).detach().numpy().ravel()


def get_models(dim):
    return {
        "logistic":Pipeline([("sc",StandardScaler()),("m",LogisticRegression(max_iter=2000))]),
        "rf":RandomForestClassifier(n_estimators=400),
        "svm":Pipeline([("sc",StandardScaler()),("m",SVC(probability=True))]),
        "pca_svm":Pipeline([("pca",PCA(30)),("sc",StandardScaler()),("m",SVC(probability=True))]),
        "xgb":XGBClassifier(n_estimators=400,eval_metric="auc"),
        "lgb":LGBMClassifier(n_estimators=500),
        "dnn":DNNClassifier(dim),
        "bagging":BaggingClassifier(LogisticRegression(max_iter=2000),n_estimators=10),
        "voting":VotingClassifier([
            ("lr",LogisticRegression(max_iter=2000)),
            ("rf",RandomForestClassifier(n_estimators=300)),
            ("svm",SVC(probability=True))
        ],voting="soft"),
        "stacking":StackingClassifier(
            estimators=[
                ("lr",LogisticRegression(max_iter=2000)),
                ("rf",RandomForestClassifier(n_estimators=300))
            ],
            final_estimator=LogisticRegression(max_iter=2000)
        ),
        "ga_xgb":XGBClassifier(n_estimators=500,max_depth=5,learning_rate=0.05,eval_metric="auc"),
        "pso_svm":Pipeline([("sc",StandardScaler()),("m",SVC(C=10,gamma=0.01,probability=True))]),
        "bayes_opt_svms":Pipeline([("sc",StandardScaler()),("m",SVC(C=5,gamma=0.05,probability=True))])
    }


data=DataLoader().load()
pb=PairBuilder(data.D,data.S,data.A)
pos,neg=pb.split()
neg_feats=[pb.feat(r,d) for r,d in neg]

pdf_path=f"{cfg.OUTPUT_DIR}/CLUSTER_5_TO_25_REPORT.pdf"
rows=[]

with PdfPages(pdf_path) as pdf:
    for k in cfg.CLUSTER_RANGE:
        sampler=ClusterSampler(k)
        neg_sampled=sampler.sample(neg,neg_feats,len(pos))
        X,y=DatasetBuilder(pb).build(neg_sampled,pos)
        skf=StratifiedKFold(cfg.N_SPLITS,shuffle=True,random_state=cfg.RANDOM_STATE)
        models=get_models(X.shape[1])

        for name,model in models.items():
            rocs=[]; accs=[]; f1s=[]
            for tr,te in skf.split(X,y):
                if name=="dnn":
                    model.fit(X[tr],y[tr])
                    p=model.predict_proba(X[te])
                else:
                    model.fit(X[tr],y[tr])
                    p=model.predict_proba(X[te])[:,1]
                pred=(p>=0.5).astype(int)
                rocs.append(roc_auc_score(y[te],p))
                accs.append(accuracy_score(y[te],pred))
                f1s.append(f1_score(y[te],pred))

            rows.append([k,name,np.mean(rocs),np.mean(accs),np.mean(f1s)])

        dfk=pd.DataFrame([r for r in rows if r[0]==k],columns=["clusters","model","roc_auc","accuracy","f1"])
        plt.figure(figsize=(10,6))
        plt.bar(dfk["model"],dfk["roc_auc"])
        plt.xticks(rotation=45,ha="right")
        plt.title(f"ROC-AUC Comparison (k={k})")
        plt.tight_layout()
        pdf.savefig(); plt.close()

    df=pd.DataFrame(rows,columns=["clusters","model","roc_auc","accuracy","f1"])
    plt.figure(figsize=(11,8.5)); plt.axis("off")
    plt.text(0.01,0.99,df.to_string(index=False),family="monospace",va="top")
    pdf.savefig(); plt.close()

df.to_csv(f"{cfg.OUTPUT_DIR}/comparison_clusters_5_25.csv",index=False)
print("PDF saved:",pdf_path)


[LightGBM] [Info] Number of positive: 808, number of negative: 806
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002074 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 84990
[LightGBM] [Info] Number of data points in the train set: 1614, number of used features: 446
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500620 -> initscore=0.002478
[LightGBM] [Info] Start training from score 0.002478
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, be

/home/naim/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

/home/naim/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

...

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

/home/naim/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

/home/naim/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/naim/anaconda3/lib/python3.13/site-packages/torch/autograd/graph.py:841: UserWarning: CUDA initialization: Unexpected error from cudaGetDeviceCount(). Did you run some cuda functions before calling NumCudaDevices() that might have already set an error? Error 804: forward compatibility was attempted on non supported HW (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


KeyboardInterrupt: 